# Aliasing

This notebook makes Nyquist audible. When the sample rate drops below twice the highest frequency present, the digitized signal folds into a false lower frequency instead of representing the original one.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Frequency Folding

Sampling only constrains frequency modulo the sample rate. That is why multiple analog frequencies can pass through the same sample sequence.

In [ ]:
tone_freq = 6000
duration = 1.0
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_aliasing(fs_sample=20_000):
    axes[0].clear()
    axes[1].clear()
    t_hr = np.arange(0, 0.005, 1 / 200_000)
    analog = np.cos(2 * np.pi * tone_freq * t_hr)
    t_samples = np.arange(0, duration, 1 / fs_sample)
    sampled = np.cos(2 * np.pi * tone_freq * t_samples)
    play = normalize(resample_signal(sampled, fs_sample, 44_100))
    alias_freq = abs(((tone_freq + fs_sample / 2) % fs_sample) - fs_sample / 2)

    axes[0].plot(t_hr * 1e3, analog, label="Analog")
    axes[0].stem(t_samples[:40] * 1e3, sampled[:40], linefmt="tab:red", markerfmt="ro", basefmt=" ")
    axes[0].set_xlabel("Time (ms)")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title(f"Samples at {fs_sample:.0f} Hz")
    axes[1].text(0.05, 0.7, f"Original tone: {tone_freq:.0f} Hz", transform=axes[1].transAxes, fontsize=12)
    axes[1].text(0.05, 0.5, f"Observed alias: {alias_freq:.0f} Hz", transform=axes[1].transAxes, fontsize=12)
    axes[1].text(0.05, 0.3, f"Nyquist limit: {fs_sample/2:.0f} Hz", transform=axes[1].transAxes, fontsize=12)
    axes[1].set_axis_off()
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, play, rate=44_100)

controls = widgets.interactive(
    update_aliasing,
    fs_sample=float_slider(min_value=4000, max_value=24000, step=500, value=20000, description="fs"),
)
display(controls, audio_out)


## Same Samples, Different Analog Stories

The aliasing trap is that the discrete samples alone do not tell you which high-frequency source created them. Anti-alias filtering is what removes those impossible alternatives before sampling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

def update_alias_visual(fs_sample=8000, analog_freq=6000):
    ax.clear()
    t = np.arange(0, 0.006, 1 / 200_000)
    t_samples = np.arange(0, t[-1], 1 / fs_sample)
    analog = np.cos(2 * np.pi * analog_freq * t)
    samples = np.cos(2 * np.pi * analog_freq * t_samples)
    alias_freq = abs(((analog_freq + fs_sample / 2) % fs_sample) - fs_sample / 2)
    alias_curve = np.cos(2 * np.pi * alias_freq * t)
    ax.plot(t * 1e3, analog, label="Original analog")
    ax.plot(t * 1e3, alias_curve, "--", label="Aliased interpretation")
    ax.plot(t_samples * 1e3, samples, "o", label="Samples")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.set_title("Multiple analog curves through the same samples")
    ax.legend()
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_alias_visual,
    fs_sample=float_slider(min_value=4000, max_value=16000, step=500, value=8000, description="fs"),
    analog_freq=float_slider(min_value=1000, max_value=12000, step=250, value=6000, description="f analog"),
)
display(controls)


## Key Takeaway

Aliasing is not just distortion. It is a wrong answer that looks internally consistent. That is why anti-alias filtering and Nyquist discipline matter before the ADC ever takes a sample.